# Credit card default prediction: modelling

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, precision_recall_curve,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
FEATURES_PATH = "credit_features.csv"
TRAIN_PATH = "credit_train.csv"
TEST_PATH = "credit_test.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5
TARGET = "default"
CV_SCORING = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
}
CPU_CORES = 8
os.environ["LOKY_MAX_CPU_COUNT"] = str(CPU_CORES)

## 1. Introduction

This notebook reads `credit_features.csv`, generated at the end of `01_EDA_CC`, which must be run first. It contains the 17 variables selected in feature engineering and the target `default` (1 = default in October 2005).

The goal is to train and compare default models on the training set and choose the final model. The evaluation of the chosen model (threshold, calibration, test and interpretability) and the conclusions are in `03_Evaluation_CC`, which reads the `credit_train.csv` and `credit_test.csv` files generated here.

Educational project: the data are from 2005 and from a single market, and the model is not intended for real credit decisions.

## 2. Data loading

In [ ]:
credit_features = pd.read_csv(FEATURES_PATH)

categorical_features = ["SEX", "EDUCATION", "MARRIAGE", "STATUS_1"]
numeric_features = [
    "AGE", "AGE_SQUARED_DEVIATION", "LOG_LIMIT",
    "DELAY_1", "MONTHS_DELAYED", "WORST_DELAY", "MONTHS_PAID_FULL",
    "UTILIZATION_MEAN", "PAYMENT_RATIO_MEAN", "DEBT_TREND",
    "LOG_PAYMENT_LAST", "LOG_PAYMENT_MEAN", "INACTIVE",
]
model_features = categorical_features + numeric_features

pd.Series({
    "rows": len(credit_features),
    "features": len(model_features),
    "columns match": set(model_features + [TARGET]) == set(credit_features.columns),
    "nulls": credit_features.isna().sum().sum(),
})

## 3. Train/test split

We hold out 20% of the clients as the test set. The split is stratified to keep the same proportion of defaults in both sets.

From here on, every decision (model, hyperparameters, threshold and calibration) is made with the training set only. The test set is used once, at the end, to measure performance on clients the model has never seen.

In [ ]:
X = credit_features[model_features]
y = credit_features[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

In [ ]:
pd.DataFrame({
    "clients": [len(y_train), len(y_test)],
    "defaults": [y_train.sum(), y_test.sum()],
    "default_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]).round(3)

The training set has 24,000 clients (5,309 defaults) and the test set 6,000 (1,327 defaults). Both keep the 22.1% default rate. With 1,327 defaults in the test set, the final metrics will be stable: a recall of 60%, for example, would have a confidence interval of about ±2.6 points.

We save both sets, with the target, to `credit_train.csv` and `credit_test.csv`.

In [ ]:
pd.concat([X_train, y_train], axis=1).to_csv(TRAIN_PATH, index=False)
pd.concat([X_test, y_test], axis=1).to_csv(TEST_PATH, index=False)

pd.Series({
    "credit_train.csv": len(pd.read_csv(TRAIN_PATH)),
    "credit_test.csv": len(pd.read_csv(TEST_PATH)),
}, name="rows")

## 4. Cross-validation

We split the training set into 5 folds of about 4,800 clients, all with the same proportion of defaults (22.1%): that is what "stratified" means. We train 5 times, each time leaving a different fold out for evaluation, and average the results. This lets us compare models and hyperparameters without touching the test set.

In [ ]:
cross_validation = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

## 5. Baseline model: logistic regression

Logistic regression is the standard model banks use for this kind of task. It is stable and its coefficients read as effects on risk. It sets the bar that more complex models will have to beat.

We do not rebalance the classes (`class_weight="balanced"`), because it would inflate the probabilities and worsen calibration. The imbalance is handled through the decision threshold.

### 5.1 Pipeline

- **Categorical variables**: one-hot encoding with `drop="first"`. Each coefficient is read relative to a reference category (the first in alphabetical order): `man`, `high_school`, `divorced` and `paid_full`.
- **Numeric variables**: standardised, so that coefficients are comparable with each other (effect of one standard deviation).

In [ ]:
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(drop="first"), categorical_features),
    ("numeric", StandardScaler(), numeric_features),
])

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

### 5.2 Regularisation

We try four values of `C` and keep the one that maximises the mean PR-AUC in our cross-validation.

In [ ]:
logistic_search = GridSearchCV(
    logistic_pipeline,
    param_grid={"classifier__C": [0.01, 0.1, 1, 10]},
    scoring="average_precision",
    cv=cross_validation,
)
logistic_search.fit(X_train, y_train)

search_results = pd.DataFrame(logistic_search.cv_results_)
search_results[["param_classifier__C", "mean_test_score", "std_test_score"]].round(4)

Regularisation barely matters: PR-AUC goes from 0.532 to 0.535, a difference smaller than the variation across folds (±0.012). The search picks `C = 10`.

### 5.3 Cross-validation metrics

We evaluate the chosen model with ROC-AUC, PR-AUC and the Brier score, which do not depend on the decision threshold.

Notes:
- ROC-AUC measures the model's ability to separate positive and negative cases across all possible decision thresholds.
- PR-AUC is useful on an imbalanced dataset like this one. Unlike ROC-AUC, it ignores true negatives and focuses on the positive class.
- The Brier score measures the accuracy of the predicted probabilities. The closer to 0, the closer to a perfect prediction.
- For reference, a model with no predictive power would have a ROC-AUC of 0.5 and a PR-AUC equal to the default rate (0.221). Always predicting the default rate would give a Brier score of 0.221 × 0.779 = 0.172.

In [ ]:
best_logistic = logistic_search.best_estimator_
logistic_cv = cross_validate(
    best_logistic, X_train, y_train, cv=cross_validation, scoring=CV_SCORING
)

logistic_metrics = pd.DataFrame({
    "mean": [
        logistic_cv["test_roc_auc"].mean(),
        logistic_cv["test_pr_auc"].mean(),
        -logistic_cv["test_brier"].mean(),
    ],
    "std": [
        logistic_cv["test_roc_auc"].std(),
        logistic_cv["test_pr_auc"].std(),
        logistic_cv["test_brier"].std(),
    ],
}, index=["ROC-AUC", "PR-AUC", "Brier score"])
logistic_metrics.round(4)

The model has moderate and stable predictive power across folds:

| Metric | Logistic | No predictive power |
|---|---|---|
| ROC-AUC | 0.778 (±0.007) | 0.500 |
| PR-AUC | 0.535 (±0.012) | 0.221 |
| Brier score | 0.137 (±0.002) | 0.172 |

- A **ROC-AUC of 0.78** means that, if we pick at random one client who defaults and one who does not, the model assigns more risk to the first one in 78% of cases.
- A **PR-AUC of 0.535** is 2.4 times that of a model with no predictive power.
- The **Brier score** reduces by 21% the error of always predicting the average rate.

This is the bar the tree models in section 6 will have to beat.

### 5.4 Coefficients

With standardised numeric variables, each coefficient measures the effect on default risk on the log-odds scale (the logarithm of the odds). Positive means more risk.

In [ ]:
coefficient_names = best_logistic.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.Series(
    best_logistic.named_steps["classifier"].coef_[0], index=coefficient_names
).sort_values()

ax = coefficients.plot(kind="barh", figsize=(8, 8))
ax.axvline(0, color="grey", linewidth=1)
ax.set_title("Logistic regression coefficients")
ax.set_xlabel("Coefficient (log-odds scale)")
ax.set_ylabel("Variable")
plt.show()

Some notes:
- **Delay matters most:** `DELAY_1`, `MONTHS_DELAYED` and `WORST_DELAY` increase risk. The negative coefficient of `STATUS_1_past_due` does not contradict this: the effect of delay is split between that category and `DELAY_1`, and overall it is clearly positive.
- **We also see some expected behaviours.** Paying more and having a higher limit reduce risk, and using more of the limit increases it.
- **Demographic variables weigh little** once payment behaviour is taken into account.

## 6. Tree models

We try two models that are more flexible than logistic regression and can capture non-linear relationships and interactions between variables:

- **Random Forest**: many independent trees whose predictions are averaged.
- **Gradient boosting** (`HistGradientBoostingClassifier`): small trees trained in sequence, each correcting the errors of the previous one.

We set a criterion: a tree model is only worth it if it improves the PR-AUC of the logistic regression.

We use PR-AUC because the dataset is imbalanced and the class we care about is the positive one. This way the metric is not inflated by the many negatives.

In [ ]:
tree_preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(), categorical_features),
    ("numeric", "passthrough", numeric_features),
])

We tune each model with `GridSearchCV`, a small grid and the same folds as the logistic regression, and keep the combination with the best PR-AUC. The `best_cv_metrics` function summarises the validation metrics of that combination.

In [ ]:
def best_cv_metrics(search):
    best_row = pd.DataFrame(search.cv_results_).iloc[search.best_index_]
    return pd.Series({
        "ROC-AUC": best_row["mean_test_roc_auc"],
        "PR-AUC": best_row["mean_test_pr_auc"],
        "PR-AUC std": best_row["std_test_pr_auc"],
        "Brier score": -best_row["mean_test_brier"],
    })

### 6.1 Random Forest

We try the maximum tree depth and the minimum number of clients per leaf, the two parameters that most control overfitting, with 300 trees.

In [ ]:
forest_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE
    )),
])

forest_search = GridSearchCV(
    forest_pipeline,
    param_grid={
        "classifier__max_depth": [8, 12, None],
        "classifier__min_samples_leaf": [20, 50],
    },
    scoring=CV_SCORING,
    refit="pr_auc",
    cv=cross_validation,
    return_train_score=True,
)
forest_search.fit(X_train, y_train)
forest_search.best_params_

### 6.2 Gradient boosting

We try the learning rate and the maximum number of leaves per tree. Early stopping stops adding trees when the model no longer improves, which limits overfitting.

In [ ]:
boosting_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        max_iter=500, early_stopping=True, random_state=RANDOM_STATE
    )),
])

boosting_search = GridSearchCV(
    boosting_pipeline,
    param_grid={
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_leaf_nodes": [15, 31],
    },
    scoring=CV_SCORING,
    refit="pr_auc",
    cv=cross_validation,
    return_train_score=True,
)
boosting_search.fit(X_train, y_train)
boosting_search.best_params_

### 6.3 Comparison with logistic regression

Cross-validation metrics of the three models, with the same five folds.

In [ ]:
model_comparison = pd.DataFrame({
    "Logistic regression": {
        "ROC-AUC": logistic_metrics.loc["ROC-AUC", "mean"],
        "PR-AUC": logistic_metrics.loc["PR-AUC", "mean"],
        "PR-AUC std": logistic_metrics.loc["PR-AUC", "std"],
        "Brier score": logistic_metrics.loc["Brier score", "mean"],
    },
    "Random Forest": best_cv_metrics(forest_search),
    "Gradient boosting": best_cv_metrics(boosting_search),
}).T
model_comparison[["ROC-AUC", "PR-AUC", "PR-AUC std", "Brier score"]].round(4)

| Model | ROC-AUC | PR-AUC | Brier score |
|---|---|---|---|
| Logistic regression | 0.778 | 0.535 | 0.137 |
| Random Forest | 0.787 | 0.563 | 0.133 |
| Gradient boosting | 0.785 | 0.559–0.561 | 0.133 |

Both tree models improve the PR-AUC of the logistic regression. Between them, however, there is a tie.

### 6.4 Hits and errors of each model

To count hits and errors, each client needs a probability and a threshold that turns it into a decision (1 = flagged as default risk).

We use **out-of-fold** predictions (`cross_val_predict`): each training client gets the probability from the model trained on the other four folds. This way we evaluate all 24,000 clients without touching the test set.

In [ ]:
candidate_models = {
    "Logistic regression": best_logistic,
    "Random Forest": forest_search.best_estimator_,
    "Gradient boosting": boosting_search.best_estimator_,
}

out_of_fold_probabilities = {}
for model_name, model in candidate_models.items():
    probabilities = cross_val_predict(
        model, X_train, y_train, cv=cross_validation, method="predict_proba"
    )
    out_of_fold_probabilities[model_name] = probabilities[:, 1]

The `threshold_metrics` function counts, for a given threshold, the defaults detected (true positives), the defaults missed (false negatives) and the good clients flagged (false positives), and computes precision, recall and F1.

In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    true_negatives, false_positives, false_negatives, true_positives = confusion_matrix(
        y_true, predictions
    ).ravel()
    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    return {
        "threshold": threshold,
        "% flagged": predictions.mean() * 100,
        "defaults detected (TP)": true_positives,
        "defaults missed (FN)": false_negatives,
        "good clients flagged (FP)": false_positives,
        "precision": precision,
        "recall": recall,
        "F1": 2 * precision * recall / (precision + recall),
    }

For a fair comparison, all models flag **the same share of clients**: the 5%, 10%, 20% or 30% riskiest according to each model.

In [ ]:
rejection_rates = [0.05, 0.10, 0.20, 0.30]

results_by_rate = {}
for rate in rejection_rates:
    for model_name, probabilities in out_of_fold_probabilities.items():
        threshold = np.quantile(probabilities, 1 - rate)
        metrics = threshold_metrics(y_train, probabilities, threshold)
        results_by_rate[(f"{rate:.0%} flagged", model_name)] = metrics

pd.DataFrame(results_by_rate).T.round(3)

Since all models flag the same number of clients, every extra default a model detects is also one fewer good client flagged by mistake. It is enough to look at the defaults detected (out of 5,309) and the precision:

| Clients flagged | Logistic | Random Forest | Gradient boosting |
|---|---|---|---|
| 5% (1,200) | 864 · precision 0.72 | **913 · 0.76** | 907 · 0.76 |
| 10% (2,400) | 1,631 · 0.68 | **1,710 · 0.71** | 1,703 · 0.71 |
| 20% (4,800) | 2,731 · 0.57 | 2,735 · 0.57 | 2,726 · 0.57 |
| 30% (7,200) | 3,343 · 0.46 | 3,382 · 0.47 | 3,396 · 0.47 |

- **Among the 5–10% riskiest clients, the trees win:** they detect between 40 and 80 more defaults than the logistic regression (4–6% more) and make as many fewer mistakes.
- **From 20% onwards, the three models give practically the same result.**
- **Random Forest and gradient boosting are tied**, with differences of a few clients.

The gain from the trees is real but moderate, and it is concentrated in the riskiest clients. It is useful if the bank only acts on the riskiest cases, for example with manual review or a lower limit. Gradient boosting figures may vary slightly across machines.

## 7. Model choice

### 7.1 Precision-recall curves

Each point on the curve is a different threshold: on the right, low thresholds that detect many defaults (high recall) at the cost of more false alarms (low precision). A higher curve is better.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, probabilities in out_of_fold_probabilities.items():
    precision, recall, _ = precision_recall_curve(y_train, probabilities)
    ax.plot(recall, precision, label=model_name)
ax.axhline(y_train.mean(), color="grey", linestyle="--", label="No predictive power")
ax.set_title("Precision-recall curves (out-of-fold predictions)")
ax.set_xlabel("Recall (share of defaults detected)")
ax.set_ylabel("Precision (share of flagged clients who default)")
ax.legend()
plt.show()

The curves of the two tree models lie above the logistic regression mainly on the left, at low recall. That area corresponds to the riskiest clients: the trees identify them more precisely. From a recall of 0.5 the three curves practically overlap. Random Forest and gradient boosting are almost indistinguishable throughout.

### 7.2 Decision

- **Both tree models beat the logistic regression, but only slightly:** about 0.025 of PR-AUC, with the advantage concentrated in the 5–10% riskiest clients.
- **Random Forest and gradient boosting are tied.** We choose the **Random Forest** because it gives exactly the same results on any machine and is slightly better on the riskiest clients.
- **Logistic regression remains an alternative:** it loses little performance and is much easier to explain to clients and regulators.